In [1]:
import os
from sedona.spark import SedonaContext
from sedona.spark import dataframe_to_arrow
from sedona.spark.geoarrow import create_spatial_dataframe
from sedona.spark.maps.SedonaKepler import SedonaKepler

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 10:06:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/09 10:06:09 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/11/09 10:06:09 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/11/09 10:06:09 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/11/09 10:06:09 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/11/09 10:06:09 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/11/09 10:06:09 WARN SimpleFunctionRegistry: The function st_envelop

# Load input data

In [3]:
import pyspark.sql.functions as f

paris_places = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/paris_places")

# Create H3 cells for Paris

In [4]:
paris_polygon_wkt = "POLYGON((2.2241 48.8156, 2.4699 48.8156, 2.4699 48.9022, 2.2241 48.9022, 2.2241 48.8156))"

In [5]:
h3_cells = sedona.sql(
    f"""
    WITH h3_cells AS (
        SELECT
            id,
            ST_H3ToGeom(ARRAY(id))[0] AS geom
        LATERAL VIEW EXPLODE(ST_H3CellIDs(ST_GeomFromText('{paris_polygon_wkt}'), 8, true)) AS id
    )
    SELECT 
        id,
        geom,
        ST_X(ST_Centroid(geom)) AS lon,
        ST_Y(ST_Centroid(geom)) AS lat
    FROM h3_cells
    """
)

In [6]:
SedonaKepler.create_map(h3_cells, "h3_cells")

/usr/local/lib/python3.10/dist-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


KeplerGl(data={'h3_cells': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20…

In [7]:
h3_cells.createOrReplaceTempView("h3_cells")

# Filter data to categories

In [8]:
categories = [
    'restaurant',
    'shopping',
    'bakery',
    'education',
    'school',
    'pharmacy', 
    'cafe',
    'theatre',
    'transportation',
    'hospital',
    'park'
]

paris_places.\
    where(f"categories.primary IN {tuple(categories)}").\
    selectExpr("categories.primary AS category", "geometry").\
    createOrReplaceTempView("selected_categories")

In [9]:
sedona.sql("select * from selected_categories").show()

[Stage 5:>                                                          (0 + 1) / 1]

+--------------+--------------------+
|      category|            geometry|
+--------------+--------------------+
|      hospital|POINT (2.2245171 ...|
|      hospital|POINT (2.224481 4...|
|        school|POINT (2.2271923 ...|
|        school|POINT (2.2248187 ...|
|    restaurant|POINT (2.2278196 ...|
|        bakery|POINT (2.2281414 ...|
|    restaurant|POINT (2.2277854 ...|
|        bakery|POINT (2.2280888 ...|
|     education|POINT (2.2377495 ...|
|        school|POINT (2.2385001 ...|
|transportation|POINT (2.2401927 ...|
|    restaurant|POINT (2.24062 48...|
|    restaurant|POINT (2.2473512 ...|
|    restaurant|POINT (2.2476 48....|
|        bakery|POINT (2.2492841 ...|
|       theatre|POINT (2.2517361 ...|
|          park|POINT (2.2425 48....|
|    restaurant|POINT (2.2437786 ...|
|        school|POINT (2.2452141 ...|
|      shopping|POINT (2.2466481 ...|
+--------------+--------------------+
only showing top 20 rows



# Create walk catchments

In [17]:
import requests
from shapely.geometry import shape, MultiPolygon

OPEN_ROUTING_URL = "http://ors:8082"

def walk_time_polygon(lon: float, lat: float, minutes: int):
    body = {
      "locations": [[lon, lat]],
      "range": [minutes * 60],
      "range_type": "time"
    }

    response = requests.post(
        url=f"{OPEN_ROUTING_URL}/ors/v2/isochrones/foot-walking",
        json=body
    )

    if response.status_code != 200:
        return None
        
    response_data = response.json()
    features = response_data["features"]
    shapely_polygons = [
        shape(feature["geometry"]) for feature in features 
        if feature["geometry"]["type"] == 'Polygon'
    ]
    
    return MultiPolygon(shapely_polygons)

In [18]:
import pyspark.sql.functions as f
import sedona.spark.sql.types as st
import shapely.geometry.base as b
 
def create_walk_catchment(
    lon: float,
    lat: float,
    minutes: int
) -> b.BaseGeometry:
    return walk_time_polygon(lon, lat, minutes)
 
create_walk_catchment_udf = f.udf(
    create_walk_catchment,
    st.GeometryType()
)
 
sedona.udf.register(
    "ST_GetWalkCatchment",
    create_walk_catchment_udf
)

In [19]:
catchments_sample = sedona.sql(
"""
SELECT
    id,
    ST_GetWalkCatchment(lon, lat, 10) AS walk_catchment,
    geom
FROM h3_cells
"""
).limit(100)

SedonaKepler.create_map(catchments_sample, "catchments")

KeplerGl(data={'catchments': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, …

In [22]:
sedona.sql(
    """
    WITH catchments AS (
        SELECT
            id,
            ST_GetWalkCatchment(lon, lat, 10) AS walk_catchment,
            geom
        FROM h3_cells
    ),
    joined AS (
        SELECT 
            *,
            c.geom AS grid_geom
        FROM catchments AS c
        JOIN selected_categories AS S ON ST_Intersects(s.geometry, c.walk_catchment)
    )
    SELECT
        id,
        category,
        count(*) AS count,
        FIRST(grid_geom) AS geom
    FROM joined
    GROUP BY id, category
    """
).createOrReplaceTempView("catchments_count")

In [23]:
sedona.sql("SELECT * FROM catchments_count").show(10)

[Stage 21:==============================================>           (4 + 1) / 5]

+------------------+--------------+-----+--------------------+
|                id|      category|count|                geom|
+------------------+--------------+-----+--------------------+
|613047287637082111|     education|    4|POLYGON ((2.41381...|
|613047287637082111|          park|    1|POLYGON ((2.41381...|
|613047287637082111|    restaurant|    3|POLYGON ((2.41381...|
|613047287637082111|      shopping|    2|POLYGON ((2.41381...|
|613047287637082111|transportation|    2|POLYGON ((2.41381...|
|613047287639179263|     education|    3|POLYGON ((2.42511...|
|613047287639179263|      pharmacy|    1|POLYGON ((2.42511...|
|613047287639179263|    restaurant|    6|POLYGON ((2.42511...|
|613047287639179263|        school|    1|POLYGON ((2.42511...|
|613047287639179263|      shopping|    3|POLYGON ((2.42511...|
+------------------+--------------+-----+--------------------+
only showing top 10 rows



# Pivot

In [28]:
feature_df = sedona.table("catchments_count").\
    groupBy("id", "geom").pivot("category", categories).\
    agg(f.first("count")).\
    selectExpr(
        "id",
        "geom",
        "COALESCE(restaurant, 0) AS restaurant",
        "COALESCE(shopping, 0) AS shopping",
        "COALESCE(bakery, 0) AS bakery",
        "COALESCE(education, 0) AS education",
        "COALESCE(school, 0) AS school",
        "COALESCE(pharmacy, 0) AS pharmacy",
        "COALESCE(cafe, 0) AS cafe",
        "COALESCE(theatre, 0) AS theatre",
        "COALESCE(transportation, 0) AS transportation",
        "COALESCE(park, 0) AS park",
        "COALESCE(hospital, 0) AS hospital"
    )

In [29]:
feature_df.\
    withColumn("id", f.concat(f.expr("substring(id, 0, 5)"), f.lit("..."))).\
    drop("school", "restaurant", "shopping", "transportation").\
    show(5)

[Stage 32:==============================================>           (4 + 1) / 5]

+--------+--------------------+------+---------+--------+----+-------+----+--------+
|      id|                geom|bakery|education|pharmacy|cafe|theatre|park|hospital|
+--------+--------------------+------+---------+--------+----+-------+----+--------+
|61304...|POLYGON ((2.35032...|    52|       50|      26|  42|     53|  12|      20|
|61304...|POLYGON ((2.26893...|     5|        3|       7|   3|      1|   5|       1|
|61304...|POLYGON ((2.36062...|    16|       17|      14|  15|     23|   6|       3|
|61304...|POLYGON ((2.47322...|     0|        1|       0|   0|      0|   0|       1|
|61304...|POLYGON ((2.43537...|     0|        0|       0|   0|      8|   4|       0|
+--------+--------------------+------+---------+--------+----+-------+----+--------+
only showing top 5 rows



# Kmeans

## Create features vector

In [37]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.clustering import KMeans

assembler = VectorAssembler(
    inputCols=categories,
    outputCol="features",
)

assembled_df = assembler.transform(feature_df)

scaler = MinMaxScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scaled_data = scaler.fit(assembled_df).transform(assembled_df)

In [41]:
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, DenseVector
from pyspark.sql.types import ArrayType, DoubleType

@udf(ArrayType(DoubleType()))
def round_vector(v):
    if v is None:
        return None
    return [round(float(x), 4) for x in v]  # 4 decimal places

# Apply it
df_rounded = scaled_data.withColumn("scaled_rounded", round_vector("scaled_features"))

df_rounded.select("id", "scaled_rounded").show(5)

[Stage 121:=============================================>           (4 + 1) / 5]

+------------------+--------------------+
|                id|      scaled_rounded|
+------------------+--------------------+
|613047304085045247|[1.0, 0.9548, 0.7...|
|613047302585581567|[0.023, 0.0323, 0...|
|613047303986479103|[0.2204, 0.1613, ...|
|613047304168931327|[0.0, 0.0, 0.0, 0...|
|613047304152154111|[0.0066, 0.0, 0.0...|
+------------------+--------------------+
only showing top 5 rows



In [32]:
scaled_data.printSchema()

root
 |-- id: long (nullable = true)
 |-- geom: geometry (nullable = true)
 |-- restaurant: long (nullable = false)
 |-- shopping: long (nullable = false)
 |-- bakery: long (nullable = false)
 |-- education: long (nullable = false)
 |-- school: long (nullable = false)
 |-- pharmacy: long (nullable = false)
 |-- cafe: long (nullable = false)
 |-- theatre: long (nullable = false)
 |-- transportation: long (nullable = false)
 |-- park: long (nullable = false)
 |-- hospital: long (nullable = false)
 |-- features: vector (nullable = true)
 |-- scaledFeatures: vector (nullable = true)



## Train and predict

In [43]:
# Train KMeans model
kmeans = KMeans(k=10, seed=42, featuresCol="scaled_features")
model = kmeans.fit(scaled_data)

# Make predictions
predictions = model.transform(scaled_data)
predictions.select("id", "scaled_features", "prediction").show(5)

# Show cluster centers
print("Cluster Centers:")
for center in model.clusterCenters():
    print(center)

[Stage 240:=============================================>           (4 + 1) / 5]

+------------------+--------------------+----------+
|                id|     scaled_features|prediction|
+------------------+--------------------+----------+
|613047304085045247|[1.0,0.9548387096...|         5|
|613047302585581567|[0.02302631578947...|         0|
|613047303986479103|[0.22039473684210...|         1|
|613047304168931327|(11,[3,4,9],[0.01...|         4|
|613047304152154111|(11,[0,4,7,10],[0...|         0|
+------------------+--------------------+----------+
only showing top 5 rows

Cluster Centers:
[0.03640351 0.02247312 0.04782609 0.03292683 0.05233333 0.07572464
 0.01836158 0.04121212 0.04356725 0.05930233 0.12066667]
[0.16034272 0.12828207 0.22918773 0.1704481  0.24093023 0.26744186
 0.11982657 0.17843552 0.15095879 0.18171985 0.36930233]
[0.62664474 0.77419355 0.46014493 0.85060976 0.84       0.75543478
 0.50847458 0.5        0.8245614  0.38372093 0.27      ]
[0.44465944 0.44174573 0.56265985 0.26470588 0.37647059 0.43606138
 0.49750748 0.41925134 0.2373581  0.235294

In [ ]:
SedonaKepler.create_map(predictions, "predictions")